# NanoGPT — Decoder-Only Transformer from Scratch
**Dataset:** Tiny Shakespeare | **Goal:** Understand + extend + make CV-ready

---

## What we build, step by step

```
STEP 1  Data & Tokeniser       character-level vocab, encode/decode
STEP 2  Bigram Baseline        simplest possible language model (our floor)
STEP 3  Self-Attention         the core mechanism, built from scratch
STEP 4  Multi-Head Attention   parallel attention heads
STEP 5  Transformer Block      attention + FFN + residuals + LayerNorm
STEP 6  Full GPT Model         stack of blocks + token/position embeddings
STEP 7  Training Loop          AdamW + cosine LR decay + gradient clipping
STEP 8  Ablation Experiments   depth / heads / context → perplexity table
STEP 9  Results & Plots        loss curves, perplexity, generated samples
```

## Why each piece exists

| Component | Why it exists |
|-----------|---------------|
| Token embedding | Maps integer token → continuous vector the network can process |
| Position embedding | Tells the model WHERE in the sequence each token is (attention has no order by default) |
| Masked self-attention | Each token can only attend to PAST tokens — prevents cheating by looking ahead |
| Multi-head | Run attention in parallel subspaces — each head learns different relationships |
| FFN after attention | Attention mixes information; FFN transforms it — two separate operations |
| Residual connections | Gradients flow directly to early layers — solves vanishing gradient |
| Layer normalisation | Stabilises training by normalising activations before each sub-layer |
| Dropout | Regularisation — randomly zeroes activations to prevent co-adaptation |


## Step 0 — Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import time, math, os, json
from dataclasses import dataclass
from typing import Optional

# Reproducibility
torch.manual_seed(1337)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Step 1 — Data and Character-Level Tokeniser

### What is a tokeniser?
A tokeniser converts raw text (strings) into integers the model can process.  
We use **character-level** tokenisation — each unique character becomes one token.

```
Vocabulary:  ['\n', ' ', '!', '$', ...., 'z']   (65 characters)
Encode:  'Hi' → [20, 47]
Decode:  [20, 47] → 'Hi'
```

Modern GPT models use BPE (Byte Pair Encoding) which merges common character pairs  
into single tokens — more efficient but the same idea.

### Train / Validation split
We use **90% train, 10% validation** — never train on validation data.  
Validation loss tells us if the model is generalising or memorising.

In [ ]:
# ── Download Tiny Shakespeare ─────────────────────────────────────────────
if not os.path.exists('input.txt'):
    import urllib.request
    url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    urllib.request.urlretrieve(url, 'input.txt')
    print('Downloaded Tiny Shakespeare')

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f'Total characters: {len(text):,}')
print(f'First 200 chars:\n{text[:200]}')

In [ ]:
# ── Character-level tokeniser ─────────────────────────────────────────────

# All unique characters in the dataset — this is our vocabulary
chars  = sorted(list(set(text)))
vocab_size = len(chars)
print(f'Vocabulary size: {vocab_size}')
print(f'Characters: {repr("".join(chars))}')

# stoi = string to integer, itos = integer to string
# These are the simplest possible tokeniser lookup tables
stoi = {ch: i for i, ch in enumerate(chars)}   # 'a' → 0, 'b' → 1, ...
itos = {i: ch for i, ch in enumerate(chars)}   # 0 → 'a', 1 → 'b', ...

def encode(s):
    """String → list of integers."""
    return [stoi[c] for c in s]

def decode(l):
    """List of integers → string."""
    return ''.join([itos[i] for i in l])

# Verify round-trip
test_str = 'Hello, World!'
assert decode(encode(test_str)) == test_str
print(f'\nEncode demo: "{test_str}" → {encode(test_str)}')
print(f'Decode demo: {encode(test_str)} → "{decode(encode(test_str))}"')

# ── Encode full dataset into a tensor ────────────────────────────────────
# Store as int64 on CPU first; we'll move batches to GPU during training
data = torch.tensor(encode(text), dtype=torch.long)
print(f'\nData tensor shape: {data.shape}')
print(f'Data tensor dtype: {data.dtype}')

# ── Train / validation split ──────────────────────────────────────────────
n          = int(0.9 * len(data))
train_data = data[:n]   # first 90%
val_data   = data[n:]   # last 10%
print(f'\nTrain tokens: {len(train_data):,}')
print(f'Val tokens:   {len(val_data):,}')

## Step 2 — Bigram Baseline Model

### What is a bigram model?
The simplest possible language model: **predict the next character based only on the current character.**

```
Input:  'H'
Output: probability distribution over all 65 characters
        'e' gets highest probability (H is often followed by e)
```

It's just an embedding lookup — one row per input token, each row = logits for next token.  
This gives us our **baseline** — NanoGPT must beat this significantly.

### Why build it?
It shares the same training loop and generation code as the full model.  
It makes the context window concept concrete: bigram context=1, GPT context=256.

In [ ]:
# ── Batch sampler ─────────────────────────────────────────────────────────

def get_batch(split, block_size=8, batch_size=32):
    """
    Sample a random batch of (input, target) pairs.

    block_size: context window — how many tokens the model sees at once
    batch_size: how many independent sequences per batch

    Example with block_size=4:
      data: [18, 47, 56, 57, 58, 1, 15, 47, 56]
      x:    [18, 47, 56, 57]   ← input tokens
      y:    [47, 56, 57, 58]   ← target = next token for each input position

    So one sequence of length 4 gives 4 training examples:
      context=[18]           → predict 47
      context=[18,47]        → predict 56
      context=[18,47,56]     → predict 57
      context=[18,47,56,57]  → predict 58
    """
    data_src = train_data if split == 'train' else val_data

    # Random starting positions — each at least block_size from end
    ix = torch.randint(len(data_src) - block_size, (batch_size,))

    # Stack sequences into (batch_size, block_size) tensors
    x  = torch.stack([data_src[i:i+block_size]   for i in ix])
    y  = torch.stack([data_src[i+1:i+block_size+1] for i in ix])
    return x.to(DEVICE), y.to(DEVICE)


@torch.no_grad()
def estimate_loss(model, eval_iters=200, block_size=8, batch_size=32):
    """
    Estimate mean loss over eval_iters batches.
    @torch.no_grad() — no gradient tracking needed, saves memory.
    model.eval() — disables dropout for consistent evaluation.
    """
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split, block_size, batch_size)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


# ── Bigram Model ──────────────────────────────────────────────────────────

class BigramLanguageModel(nn.Module):
    """
    Simplest language model: lookup table of shape (vocab_size, vocab_size)
    Each row = logits for next token given current token.
    No context beyond the current character.
    """
    def __init__(self, vocab_size):
        super().__init__()
        # token_embedding_table[i] = logits when current token is i
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx:     (B, T) — batch of token sequences
        # targets: (B, T) — the correct next token at each position

        # Lookup: each token → its row of logits
        logits = self.token_embedding_table(idx)   # (B, T, vocab_size)

        if targets is None:
            return logits, None

        # Reshape for cross-entropy:
        # cross_entropy expects (N, C) not (B, T, C)
        B, T, C = logits.shape
        logits  = logits.view(B*T, C)    # (B*T, vocab_size)
        targets = targets.view(B*T)      # (B*T,)

        # Cross-entropy = -log(softmax(logits)[correct_class])
        # This is the standard language modelling objective
        loss = F.cross_entropy(logits, targets)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        """
        Autoregressively generate max_new_tokens new tokens.
        idx: (B, T) current context
        """
        for _ in range(max_new_tokens):
            # Get predictions for current sequence
            logits, _ = self(idx)

            # Focus only on LAST time step — that's the next token prediction
            logits = logits[:, -1, :]           # (B, vocab_size)

            # Convert logits to probabilities
            probs  = F.softmax(logits, dim=-1)  # (B, vocab_size)

            # Sample from the distribution (not argmax — that's greedy and repetitive)
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)

            # Append sampled token to the running sequence
            idx = torch.cat((idx, idx_next), dim=1)  # (B, T+1)

        return idx


# Train bigram baseline
bigram_model = BigramLanguageModel(vocab_size).to(DEVICE)
optimizer    = torch.optim.AdamW(bigram_model.parameters(), lr=1e-3)

print('Training bigram baseline...')
for step in range(10000):
    xb, yb = get_batch('train', block_size=8, batch_size=32)
    logits, loss = bigram_model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

bigram_losses = estimate_loss(bigram_model, block_size=8, batch_size=32)
bigram_val_ppl = math.exp(bigram_losses['val'])
print(f'Bigram  train loss: {bigram_losses["train"]:.4f} | '
      f'val loss: {bigram_losses["val"]:.4f} | '
      f'val perplexity: {bigram_val_ppl:.1f}')

# Generate from bigram
context = torch.zeros((1,1), dtype=torch.long, device=DEVICE)
bigram_sample = decode(bigram_model.generate(context, max_new_tokens=200)[0].tolist())
print(f'\nBigram sample (random, low quality expected):\n{bigram_sample}')

## Step 3 — Self-Attention: The Core Mechanism

### The fundamental idea
Every token produces three vectors:
- **Query (Q):** "What am I looking for?"
- **Key (K):**   "What do I contain?"
- **Value (V):** "What information do I pass on if attended to?"

Attention score between token i and token j = `dot(Q_i, K_j)` — how relevant is j to i?

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

### Why divide by √d_k?
Without it, dot products grow large as dimension increases → softmax becomes peaky → gradients vanish.  
Dividing by `√head_size` keeps dot products at unit variance.

### Why masking?
This is a **decoder** — it generates text left to right.  
Token at position 3 must not see tokens at positions 4, 5, ... (that's future information).  
We mask those positions with `-inf` before softmax → they become 0 after softmax.

In [ ]:
# ── Single Head of Self-Attention ─────────────────────────────────────────

class Head(nn.Module):
    """
    One head of masked self-attention.

    head_size: dimension of Q, K, V projections
    n_embd:    input embedding dimension
    block_size: maximum context length (for the causal mask)
    """
    def __init__(self, head_size, n_embd, block_size, dropout=0.1):
        super().__init__()

        # Three linear projections — no bias (standard in transformers)
        # Each takes an n_embd-dim token vector and projects to head_size-dim
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        # Causal mask: lower-triangular matrix of ones
        # tril[i,j] = 1 if j <= i (past and present), 0 if j > i (future)
        # register_buffer: not a parameter (not trained), but saved with the model
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape   # batch, time (sequence length), channels (n_embd)

        # Compute Q, K, V for every token in the sequence
        k = self.key(x)     # (B, T, head_size)
        q = self.query(x)   # (B, T, head_size)

        # ── Compute attention scores ───────────────────────────────────────
        # Q @ K^T: for each query, compute dot product with all keys
        # (B, T, hs) @ (B, hs, T) → (B, T, T)
        # wei[b, i, j] = how much token i attends to token j in batch b
        head_size = k.shape[-1]
        wei = q @ k.transpose(-2, -1) * head_size**-0.5   # scaled dot-product

        # ── Apply causal mask ─────────────────────────────────────────────
        # Replace future positions with -inf so softmax makes them 0
        # [:T, :T] slice handles sequences shorter than block_size
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

        # ── Softmax: normalise attention scores to probabilities ──────────
        wei = F.softmax(wei, dim=-1)   # (B, T, T)  — rows sum to 1
        wei = self.dropout(wei)        # randomly zero some attention weights

        # ── Weighted aggregation of values ────────────────────────────────
        v   = self.value(x)            # (B, T, head_size)
        out = wei @ v                  # (B, T, T) @ (B, T, hs) → (B, T, hs)
        # out[b, i] = weighted sum of value vectors, weighted by how much
        # token i attends to each other token
        return out


print('Self-attention Head defined.')
# Quick shape check
_h = Head(16, 32, 64).to(DEVICE)
_x = torch.randn(2, 10, 32).to(DEVICE)  # batch=2, seq=10, embd=32
_o = _h(_x)
print(f'  Input shape:  {_x.shape}')
print(f'  Output shape: {_o.shape}  (same B,T; head_size channels)')

## Step 4 — Multi-Head Attention

### Why multiple heads?
A single attention head can only learn one type of relationship between tokens.  
With multiple heads running **in parallel**, each head can specialise:
- Head 1 might learn syntactic relationships (subject → verb)
- Head 2 might learn coreference (pronoun → noun it refers to)
- Head 3 might learn positional patterns

The outputs of all heads are concatenated and projected back to `n_embd` dimensions.

```
n_embd = 64,  n_heads = 4
head_size = 64 // 4 = 16  (each head works in a 16-dim subspace)
Concat all heads: 4 × 16 = 64 → project back to 64
```

In [ ]:
# ── Multi-Head Self-Attention ─────────────────────────────────────────────

class MultiHeadAttention(nn.Module):
    """
    Multiple heads of self-attention running in parallel.

    n_heads:   number of parallel attention heads
    head_size: dimension per head = n_embd // n_heads
    """
    def __init__(self, n_heads, head_size, n_embd, block_size, dropout=0.1):
        super().__init__()

        # Create n_heads independent attention heads
        self.heads = nn.ModuleList([
            Head(head_size, n_embd, block_size, dropout)
            for _ in range(n_heads)
        ])

        # Projection: concatenated heads (n_heads * head_size = n_embd) → n_embd
        # This mixes information from all heads
        self.proj    = nn.Linear(n_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Run all heads independently, then concatenate along channel dim
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        # out shape: (B, T, n_heads * head_size) = (B, T, n_embd)

        # Project back to n_embd and apply dropout
        out = self.dropout(self.proj(out))  # (B, T, n_embd)
        return out


print('MultiHeadAttention defined.')
_mha = MultiHeadAttention(4, 8, 32, 64).to(DEVICE)
_x   = torch.randn(2, 10, 32).to(DEVICE)
_o   = _mha(_x)
print(f'  Input:  {_x.shape}')
print(f'  Output: {_o.shape}  (same shape as input — n_embd preserved)')

## Step 5 — Transformer Block

### What is a block?
One complete Transformer layer = Multi-Head Attention + Feed-Forward Network + Residuals + LayerNorm.

```
x → LayerNorm → MultiHeadAttention → + x  (residual)
              → LayerNorm → FFN         → + x  (residual)
```

### Feed-Forward Network (FFN)
After attention (which **mixes** information across tokens), the FFN **transforms** that information:  
`Linear(n_embd → 4*n_embd) → ReLU → Linear(4*n_embd → n_embd)`  
The 4× expansion is from the original *Attention is All You Need* paper.

### Residual connections
`output = sublayer(x) + x` — the input is added directly to the sublayer output.  
This means gradients can flow **directly** from the loss to any layer without passing through the whole network.

### Pre-norm vs post-norm
Original transformer: LayerNorm **after** the sublayer (post-norm).  
GPT: LayerNorm **before** the sublayer (pre-norm) — more stable training.

In [ ]:
# ── Feed-Forward Network ──────────────────────────────────────────────────

class FeedForward(nn.Module):
    """
    Position-wise FFN: applied independently to each token.
    Expands to 4*n_embd (following the original paper), then contracts back.
    This is where token representations are transformed after attention mixing.
    """
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),  # expand
            nn.ReLU(),                        # non-linearity
            nn.Linear(4 * n_embd, n_embd),  # contract
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)   # (B, T, n_embd) → (B, T, n_embd)


# ── Transformer Block ─────────────────────────────────────────────────────

class Block(nn.Module):
    """
    One Transformer decoder block:
      x = x + MultiHeadAttention(LayerNorm(x))   ← communication between tokens
      x = x + FFN(LayerNorm(x))                  ← transformation of each token

    Pre-LayerNorm formulation (GPT-2 style) — more stable than post-norm.
    Residual connections (+x) allow gradient flow to early layers.
    """
    def __init__(self, n_embd, n_heads, block_size, dropout=0.1):
        super().__init__()
        head_size = n_embd // n_heads

        # Communication sub-layer: tokens exchange information
        self.sa  = MultiHeadAttention(n_heads, head_size, n_embd, block_size, dropout)

        # Computation sub-layer: each token transforms its own representation
        self.ffn = FeedForward(n_embd, dropout)

        # Layer normalisation applied BEFORE each sub-layer (pre-norm)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # Residual + attention
        x = x + self.sa(self.ln1(x))
        # Residual + FFN
        x = x + self.ffn(self.ln2(x))
        return x


print('FeedForward and Block defined.')
_b = Block(64, 4, 128).to(DEVICE)
_x = torch.randn(2, 32, 64).to(DEVICE)   # batch=2, seq=32, embd=64
_o = _b(_x)
print(f'  Block input shape:  {_x.shape}')
print(f'  Block output shape: {_o.shape}  (shape preserved through block)')

## Step 6 — Full GPT Model

### Architecture overview
```
Input tokens  (B, T)
     ↓
Token Embedding   (B, T, n_embd)   — what is each token?
     +
Position Embedding (B, T, n_embd)  — where is each token?
     ↓
Dropout
     ↓
Block 1  (MultiHeadAttn + FFN + residuals + LN)
Block 2
  ...
Block n_layers
     ↓
Final LayerNorm
     ↓
Linear head  (n_embd → vocab_size)   — predict next token
     ↓
Logits  (B, T, vocab_size)
```

### Why both token AND position embeddings?
Attention is **permutation invariant** — `{A, B, C}` and `{C, A, B}` give the same attention scores.  
Position embeddings break this symmetry: they add information about each token's position.

In [ ]:
# ── GPT Configuration ─────────────────────────────────────────────────────

@dataclass
class GPTConfig:
    """
    All hyperparameters in one place.
    Using a dataclass makes it easy to create variants for ablation studies.
    """
    vocab_size:  int   = 65     # number of unique characters
    block_size:  int   = 256    # context window (max sequence length)
    n_embd:      int   = 384    # embedding dimension
    n_heads:     int   = 6      # number of attention heads
    n_layers:    int   = 6      # number of transformer blocks
    dropout:     float = 0.2    # dropout probability


# ── Full GPT Model ────────────────────────────────────────────────────────

class GPT(nn.Module):
    """
    Decoder-only Transformer language model.
    Architecture follows GPT-2 (Radford et al. 2019) at a small scale.
    """
    def __init__(self, config: GPTConfig):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            # Token embedding: integer token → n_embd-dim vector
            # wte = weight of token embedding (GPT-2 naming convention)
            wte  = nn.Embedding(config.vocab_size, config.n_embd),

            # Position embedding: position index → n_embd-dim vector
            # Learnable (not sinusoidal like original Transformer)
            # wpe = weight of position embedding
            wpe  = nn.Embedding(config.block_size, config.n_embd),

            # Input dropout
            drop = nn.Dropout(config.dropout),

            # Stack of n_layers transformer blocks
            h    = nn.ModuleList([
                Block(config.n_embd, config.n_heads,
                      config.block_size, config.dropout)
                for _ in range(config.n_layers)
            ]),

            # Final layer normalisation before the output head
            ln_f = nn.LayerNorm(config.n_embd),
        ))

        # Language model head: n_embd → vocab_size logits
        # No bias — standard for LM heads
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # Weight tying: share token embedding weights with the LM head
        # Intuition: the embedding that maps token → vector should be
        # similar to the matrix that maps vector → token probabilities
        # This reduces parameters significantly
        self.transformer.wte.weight = self.lm_head.weight

        # Initialise weights using GPT-2 style initialisation
        self.apply(self._init_weights)

    def _init_weights(self, module):
        """
        GPT-2 weight initialisation:
        - Linear layers: N(0, 0.02)
        - Embeddings:    N(0, 0.02)
        - Residual projections scaled by 1/sqrt(n_layers)
          (prevents residual stream from growing with depth)
        """
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.config.block_size, \
            f'Sequence length {T} > block_size {self.config.block_size}'

        # Position indices: [0, 1, 2, ..., T-1]
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)  # (T,)

        # Token + position embeddings
        tok_emb = self.transformer.wte(idx)  # (B, T, n_embd)
        pos_emb = self.transformer.wpe(pos)  # (T, n_embd) → broadcasts to (B, T, n_embd)

        # Sum token and position embeddings, apply dropout
        x = self.transformer.drop(tok_emb + pos_emb)  # (B, T, n_embd)

        # Pass through all transformer blocks
        for block in self.transformer.h:
            x = block(x)   # (B, T, n_embd) — shape preserved

        # Final layer norm
        x = self.transformer.ln_f(x)   # (B, T, n_embd)

        # Project to vocabulary logits
        logits = self.lm_head(x)       # (B, T, vocab_size)

        if targets is None:
            return logits, None

        # Compute cross-entropy loss
        B, T, C = logits.shape
        loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        Generate text autoregressively.

        temperature: > 1 = more random, < 1 = more focused
        top_k:       if set, only sample from top-k most probable tokens
        """
        for _ in range(max_new_tokens):
            # Crop context to block_size (can't attend beyond context window)
            idx_cond = idx[:, -self.config.block_size:]

            logits, _ = self(idx_cond)
            # Only last position matters — that's the next token prediction
            logits = logits[:, -1, :] / temperature  # (B, vocab_size)

            if top_k is not None:
                # Zero out all logits except the top-k
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')

            probs    = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx      = torch.cat((idx, idx_next), dim=1)

        return idx

    def num_parameters(self):
        return sum(p.numel() for p in self.parameters())


# Instantiate default model
config = GPTConfig(vocab_size=vocab_size)
model  = GPT(config).to(DEVICE)
print(f'GPT model created')
print(f'Parameters: {model.num_parameters():,}')
print(f'Config: n_layers={config.n_layers}, n_heads={config.n_heads}, '
      f'n_embd={config.n_embd}, block_size={config.block_size}')

# Shape check
_x  = torch.randint(0, vocab_size, (2, 32)).to(DEVICE)
_y  = torch.randint(0, vocab_size, (2, 32)).to(DEVICE)
_lg, _ls = model(_x, _y)
print(f'\nForward pass shape check:')
print(f'  Input:  {_x.shape}')
print(f'  Logits: {_lg.view(2,32,-1).shape}')
print(f'  Loss:   {_ls.item():.4f} (expected ~{math.log(vocab_size):.2f} for random init)')

## Step 7 — Training Loop with Cosine LR Decay and Gradient Clipping

### Cosine learning rate schedule
Instead of a constant LR, we decay it following a cosine curve:  
`lr = min_lr + 0.5 * (max_lr - min_lr) * (1 + cos(π * step/max_steps))`

This starts high (fast learning) and ends low (fine-tuning).  
Models trained with LR schedules consistently outperform constant-LR training.

### Gradient clipping
`torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)`  
Caps the gradient norm at 1.0 — prevents exploding gradients in deep networks.

### Perplexity
`perplexity = exp(cross_entropy_loss)`  
Intuition: perplexity of N means the model is "as confused as if choosing uniformly among N options."  
Lower = better. Random character-level model: exp(ln(65)) = 65. Good model: < 5.

In [ ]:
def get_lr(step, warmup_steps, max_steps, max_lr, min_lr):
    """
    Cosine learning rate schedule with linear warmup.

    Phase 1 (0 → warmup_steps):   linear increase from 0 to max_lr
    Phase 2 (warmup → max_steps): cosine decay from max_lr to min_lr

    Warmup prevents large updates early when model weights are random.
    """
    # Linear warmup
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    # After max_steps, hold at min_lr
    if step > max_steps:
        return min_lr
    # Cosine decay
    progress = (step - warmup_steps) / (max_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))


def train_model(model, config, max_iters=5000, eval_interval=500,
                batch_size=64, max_lr=3e-4, min_lr=3e-5,
                warmup_steps=100, grad_clip=1.0):
    """
    Full training loop with:
      - Cosine LR schedule with warmup
      - Gradient clipping
      - Periodic evaluation and logging
      - Loss and perplexity history
    """
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=max_lr,
        betas=(0.9, 0.95),   # GPT-2 betas
        weight_decay=0.1     # L2 regularisation on weights (not biases/embeddings)
    )

    train_losses, val_losses = [], []
    train_ppls,   val_ppls   = [], []
    lr_history, step_history = [], []

    model.train()
    t0 = time.time()

    for step in range(max_iters):
        # ── Set learning rate for this step ───────────────────────────────
        lr = get_lr(step, warmup_steps, max_iters, max_lr, min_lr)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr

        # ── Periodic evaluation ───────────────────────────────────────────
        if step % eval_interval == 0 or step == max_iters - 1:
            losses = estimate_loss(model,
                                   block_size=config.block_size,
                                   batch_size=batch_size)
            t1 = time.time()
            train_losses.append(losses['train'])
            val_losses.append(losses['val'])
            train_ppls.append(math.exp(losses['train']))
            val_ppls.append(math.exp(losses['val']))
            step_history.append(step)

            print(f'step {step:5d} | '
                  f'train loss {losses["train"]:.4f} (ppl {math.exp(losses["train"]):.1f}) | '
                  f'val loss {losses["val"]:.4f} (ppl {math.exp(losses["val"]):.1f}) | '
                  f'lr {lr:.2e} | '
                  f'time {t1-t0:.1f}s')
            t0 = time.time()

        # ── Forward pass ──────────────────────────────────────────────────
        xb, yb = get_batch('train',
                           block_size=config.block_size,
                           batch_size=batch_size)
        logits, loss = model(xb, yb)

        # ── Backward pass ─────────────────────────────────────────────────
        optimizer.zero_grad(set_to_none=True)   # set_to_none saves memory vs zero_grad()
        loss.backward()

        # Gradient clipping — caps gradient norm to prevent explosive updates
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        optimizer.step()
        lr_history.append(lr)

    return {
        'train_losses': train_losses,
        'val_losses':   val_losses,
        'train_ppls':   train_ppls,
        'val_ppls':     val_ppls,
        'steps':        step_history,
        'lr_history':   lr_history,
        'final_val_loss': val_losses[-1],
        'final_val_ppl':  val_ppls[-1],
    }


print('Training full GPT model...')
print('(~10-15 min on T4 GPU)')
history = train_model(model, config,
                      max_iters=5000,
                      eval_interval=500,
                      batch_size=64)
print(f'\nFinal val loss: {history["final_val_loss"]:.4f}')
print(f'Final val perplexity: {history["final_val_ppl"]:.2f}')
print(f'Bigram val perplexity was: {bigram_val_ppl:.1f}')
print(f'Improvement: {bigram_val_ppl / history["final_val_ppl"]:.1f}x better')

## Step 8 — Ablation Experiments (CV-Ready)

This is what separates your project from every other NanoGPT replication.
We systematically vary one thing at a time and measure the effect on validation perplexity.

| Experiment | What varies | What we learn |
|------------|------------|---------------|
| Model depth | n_layers: 2, 4, 6 | Does more depth always help? |
| Attention heads | n_heads: 2, 4, 6 | How much do parallel heads matter? |
| Context window | block_size: 64, 128, 256 | Does more context improve quality? |
| Embedding size | n_embd: 128, 256, 384 | Capacity vs training speed trade-off |

In [ ]:
def run_ablation(name, config_overrides, max_iters=2000, batch_size=64):
    """
    Train a model variant and return its final metrics.
    config_overrides: dict of GPTConfig fields to change from default.
    """
    cfg = GPTConfig(
        vocab_size  = vocab_size,
        block_size  = config_overrides.get('block_size', 256),
        n_embd      = config_overrides.get('n_embd',     384),
        n_heads     = config_overrides.get('n_heads',    6),
        n_layers    = config_overrides.get('n_layers',   6),
        dropout     = config_overrides.get('dropout',    0.2),
    )

    m = GPT(cfg).to(DEVICE)
    torch.manual_seed(1337)  # same seed every run for fair comparison

    t0 = time.time()
    hist = train_model(m, cfg,
                       max_iters=max_iters,
                       eval_interval=max_iters,  # only eval at end
                       batch_size=batch_size)
    elapsed = time.time() - t0

    result = {
        'name':          name,
        'n_layers':      cfg.n_layers,
        'n_heads':       cfg.n_heads,
        'n_embd':        cfg.n_embd,
        'block_size':    cfg.block_size,
        'n_params':      m.num_parameters(),
        'val_loss':      hist['final_val_loss'],
        'val_ppl':       hist['final_val_ppl'],
        'train_time_s':  elapsed,
    }
    print(f"{name:<35} val_loss={result['val_loss']:.4f}  "
          f"ppl={result['val_ppl']:.2f}  "
          f"params={result['n_params']:,}  "
          f"time={elapsed:.0f}s")
    return result


ablation_results = []
print('Running ablation study...')
print('='*80)

# ── Experiment 1: Model Depth ─────────────────────────────────────────────
print('\n[1] Model Depth (n_layers)')
for n_layers in [2, 4, 6]:
    r = run_ablation(f'n_layers={n_layers}', {'n_layers': n_layers, 'n_heads': 4, 'n_embd': 256})
    ablation_results.append(r)

# ── Experiment 2: Number of Attention Heads ───────────────────────────────
print('\n[2] Attention Heads (n_heads)')
for n_heads in [2, 4, 6]:
    r = run_ablation(f'n_heads={n_heads}', {'n_layers': 4, 'n_heads': n_heads, 'n_embd': 384})
    ablation_results.append(r)

# ── Experiment 3: Context Window ──────────────────────────────────────────
print('\n[3] Context Window (block_size)')
for block_size in [64, 128, 256]:
    r = run_ablation(f'block_size={block_size}', {'n_layers': 4, 'n_heads': 4,
                                                   'n_embd': 256, 'block_size': block_size})
    ablation_results.append(r)

# ── Experiment 4: Embedding Dimension ────────────────────────────────────
print('\n[4] Embedding Dimension (n_embd)')
for n_embd in [128, 256, 384]:
    r = run_ablation(f'n_embd={n_embd}', {'n_layers': 4, 'n_heads': 4, 'n_embd': n_embd})
    ablation_results.append(r)

In [ ]:
# ── Print Full Ablation Table ─────────────────────────────────────────────

print('\n' + '='*90)
print('ABLATION STUDY RESULTS')
print('='*90)
print(f'{"Experiment":<30} {"Layers":>6} {"Heads":>6} {"Embd":>6} {"Ctx":>6} '
      f'{"Params":>10} {"Val Loss":>9} {"Val PPL":>8}')
print('-'*90)

# Group by experiment type
groups = {
    'Model Depth':    [r for r in ablation_results if r['name'].startswith('n_layers')],
    'Attn Heads':     [r for r in ablation_results if r['name'].startswith('n_heads')],
    'Context Window': [r for r in ablation_results if r['name'].startswith('block_size')],
    'Embd Dim':       [r for r in ablation_results if r['name'].startswith('n_embd')],
}

for group_name, rows in groups.items():
    print(f'\n  {group_name}')
    for r in rows:
        best = min(rows, key=lambda x: x['val_ppl'])
        marker = '◄ best' if r == best else ''
        print(f'  {r["name"]:<28} {r["n_layers"]:>6} {r["n_heads"]:>6} '
              f'{r["n_embd"]:>6} {r["block_size"]:>6} '
              f'{r["n_params"]:>10,} {r["val_loss"]:>9.4f} '
              f'{r["val_ppl"]:>8.2f}  {marker}')

print('='*90)
print(f'\nBigram baseline perplexity: {bigram_val_ppl:.2f}')
best_overall = min(ablation_results, key=lambda x: x['val_ppl'])
print(f'Best ablation perplexity:   {best_overall["val_ppl"]:.2f}  '
      f'({best_overall["name"]})')
print(f'Full model perplexity:      {history["final_val_ppl"]:.2f}')

## Step 9 — Results Visualisation and Text Generation

In [ ]:
# ── Figure 1: Training curves ─────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('NanoGPT Training on Tiny Shakespeare', fontsize=13, fontweight='bold')

# Loss curves
ax = axes[0]
ax.plot(history['steps'], history['train_losses'], 'b-o', markersize=4, label='Train')
ax.plot(history['steps'], history['val_losses'],   'r-o', markersize=4, label='Val')
ax.axhline(bigram_losses['val'], color='gray', linestyle='--', label=f'Bigram baseline ({bigram_losses["val"]:.2f})')
ax.set_xlabel('Training Step')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('Loss Curves')
ax.legend()
ax.grid(alpha=0.3)

# Perplexity curves
ax = axes[1]
ax.plot(history['steps'], history['train_ppls'], 'b-o', markersize=4, label='Train PPL')
ax.plot(history['steps'], history['val_ppls'],   'r-o', markersize=4, label='Val PPL')
ax.axhline(bigram_val_ppl, color='gray', linestyle='--', label=f'Bigram ({bigram_val_ppl:.1f})')
ax.set_xlabel('Training Step')
ax.set_ylabel('Perplexity')
ax.set_title('Perplexity Curves')
ax.legend()
ax.grid(alpha=0.3)

# LR schedule
ax = axes[2]
ax.plot(history['lr_history'], color='green', linewidth=1.5)
ax.set_xlabel('Training Step')
ax.set_ylabel('Learning Rate')
ax.set_title('Cosine LR Schedule with Warmup')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png')

In [ ]:
# ── Figure 2: Ablation Results ────────────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Ablation Study: Validation Perplexity vs Architecture Choices',
             fontsize=13, fontweight='bold')

group_configs = [
    ('Model Depth',    'n_layers',   'n_layers', axes[0,0]),
    ('Attn Heads',     'n_heads',    'n_heads',  axes[0,1]),
    ('Context Window', 'block_size', 'block_size', axes[1,0]),
    ('Embd Dim',       'n_embd',     'n_embd',   axes[1,1]),
]

colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0']

for title, prefix, x_key, ax in group_configs:
    rows = [r for r in ablation_results if r['name'].startswith(prefix.split('_')[0])]
    xs   = [r[x_key] for r in rows]
    ppls = [r['val_ppl'] for r in rows]

    bars = ax.bar([str(x) for x in xs], ppls,
                  color=colors[:len(xs)], edgecolor='white', linewidth=1.2)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel(x_key.replace('_', ' ').title())
    ax.set_ylabel('Validation Perplexity (lower = better)')
    ax.grid(axis='y', alpha=0.3)

    for bar, val in zip(bars, ppls):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    # Highlight the best
    best_idx = ppls.index(min(ppls))
    bars[best_idx].set_edgecolor('black')
    bars[best_idx].set_linewidth(2.5)

plt.tight_layout()
plt.savefig('ablation_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ablation_results.png')

In [ ]:
# ── Text Generation with Different Temperatures ───────────────────────────

print('='*70)
print('GENERATED TEXT SAMPLES')
print('='*70)

temperatures = [0.5, 0.8, 1.0, 1.2]
context = torch.zeros((1, 1), dtype=torch.long, device=DEVICE)

for temp in temperatures:
    print(f'\n--- Temperature = {temp} '
          f'({"focused" if temp < 0.8 else "balanced" if temp <= 1.0 else "creative"}) ---')
    generated = model.generate(context.clone(),
                                max_new_tokens=300,
                                temperature=temp,
                                top_k=40)
    print(decode(generated[0].tolist()))

print('\n' + '='*70)
print('Effect of temperature:')
print('  Low (0.5):  More repetitive but grammatically safer')
print('  High (1.2): More varied and creative but can be incoherent')
print('  Optimal:    Usually 0.7-1.0 for language models')

In [ ]:
# ── Final Summary Table (CV-Ready) ────────────────────────────────────────

print('='*60)
print('FINAL RESULTS SUMMARY')
print('='*60)
print(f'Model:          NanoGPT (decoder-only Transformer)')
print(f'Dataset:        Tiny Shakespeare ({len(text):,} characters)')
print(f'Vocabulary:     {vocab_size} characters (character-level tokeniser)')
print(f'Parameters:     {model.num_parameters():,}')
print(f'Architecture:   {config.n_layers} layers, {config.n_heads} heads, '
      f'{config.n_embd}-dim, {config.block_size} context')
print()
print(f'Baseline (Bigram):')
print(f'  Val Loss:        {bigram_losses["val"]:.4f}')
print(f'  Val Perplexity:  {bigram_val_ppl:.2f}')
print()
print(f'NanoGPT (full model):')
print(f'  Val Loss:        {history["final_val_loss"]:.4f}')
print(f'  Val Perplexity:  {history["final_val_ppl"]:.2f}')
print(f'  vs Bigram:       {bigram_val_ppl / history["final_val_ppl"]:.1f}x lower perplexity')
print()
print('Ablation Findings:')

depth_rows = [r for r in ablation_results if r['name'].startswith('n_layers')]
heads_rows = [r for r in ablation_results if r['name'].startswith('n_heads')]
ctx_rows   = [r for r in ablation_results if r['name'].startswith('block_size')]
embd_rows  = [r for r in ablation_results if r['name'].startswith('n_embd')]

for group_name, rows in [('Depth', depth_rows), ('Heads', heads_rows),
                          ('Context', ctx_rows), ('Embd', embd_rows)]:
    best = min(rows, key=lambda x: x['val_ppl'])
    worst = max(rows, key=lambda x: x['val_ppl'])
    print(f'  {group_name}: best={best["name"]} (ppl={best["val_ppl"]:.2f}), '
          f'worst={worst["name"]} (ppl={worst["val_ppl"]:.2f})')

# Save results to JSON for the README
results_dict = {
    'model_config': vars(config),
    'n_params': model.num_parameters(),
    'bigram_val_ppl': bigram_val_ppl,
    'gpt_val_loss': history['final_val_loss'],
    'gpt_val_ppl': history['final_val_ppl'],
    'ablation_results': ablation_results
}
with open('results.json', 'w') as f:
    json.dump(results_dict, f, indent=2)
print('\nSaved results.json')

## What to Put on Your CV

Fill in the X values from your `results.json` after running the notebook:

---

**NanoGPT: Decoder-Only Transformer Language Model from Scratch**

- Implemented a decoder-only Transformer (NanoGPT) from scratch in PyTorch including character-level tokeniser, masked multi-head self-attention, positional embeddings, residual connections, and pre-LayerNorm — closely following the GPT-2 architecture

- Trained on Tiny Shakespeare (~1M chars); achieved validation perplexity of **X** vs bigram baseline of **Y** — a **Zx** improvement — using AdamW with cosine LR decay and gradient clipping

- Conducted **ablation study** across 4 architectural dimensions (depth 2/4/6 layers, attention heads 2/4/6, context window 64/128/256, embedding dim 128/256/384); found that [your finding, e.g. "context window had the largest impact on perplexity"]

- Demonstrated effect of sampling temperature (0.5–1.2) on generation quality; lower temperature produces grammatically coherent but repetitive text while higher temperature increases diversity at the cost of coherence

---

## Interview Questions This Project Prepares You For

| Question | Answer from this project |
|----------|-------------------------|
| Why divide by √d_k in attention? | Prevents peaky softmax as dim grows |
| What does masking do? | Prevents tokens from seeing future positions |
| Why residual connections? | Gradient flow to early layers |
| Pre-norm vs post-norm? | Pre-norm more stable; used in GPT-2 |
| Why weight tying? | Embedding and unembedding share weights — reduces params |
| What is perplexity? | exp(cross-entropy); how "surprised" the model is |
| Why cosine LR schedule? | Fast early learning, fine-tuning near end |
| What does temperature control? | Sharpness of sampling distribution |